# Airspace Object Detection: Tracking on Real Drone Video

So far every test was on single images. Real systems watch video, where a detector that fires on one frame and misses the next gives a flickering, noisy picture. This notebook adds **tracking** (ByteTrack) on top of the combined 6-class model and tests it on real drone videos from the **Anti-UAV300** dataset (visible-light videos, one drone per video, labeled every frame). The model has never seen these videos.

It compares four setups on the same frames:
1. **Detector only, 640 px** (what we've done so far)
2. **Detector + tracking, 640 px**: a box is only reported once its track has lasted 5 frames, which removes one-frame false alarms
3. **Detector only, 1280 px**: same model, larger input, to see if small, far-away drones are the problem
4. **Detector + tracking, 1280 px**

Metrics per frame: precision (reported boxes that are the drone), recall (frames where the drone is found), F1, and the **number of track IDs per video** (ideally 1, since the drone is the same object the whole time).

**How to run:** inputs are the combined-model notebook output and the `mm1991/anti-uav-dataset-300` dataset. Use GPU T4 x2, then Save Version → Save & Run All (about 20 to 30 minutes). No secrets are needed.

**v2 (after v1 results):** v1 showed the detector alone already does well on these videos (precision 0.90, recall 0.97), while plain ByteTrack *hurt* recall (0.75) and split each drone into ~8 track IDs per video. Two causes: (1) the camera pans to follow the drone, so frame-to-frame box overlap is low and tracks break, and (2) the "wait 5 frames" rule throws away the first frames of every new track. v2 drops 1280 px (it didn't help without retraining) and tests: a **hybrid filter** (report a box immediately if confidence >= 0.5, otherwise only once its track has lasted 5 frames), tuned ByteTrack (longer memory, looser matching), and **BoT-SORT with camera-motion compensation**.

In [ ]:
!pip install -q ultralytics lap

In [ ]:
# 1. Find the model and the videos
import glob, os, json, random, collections
import numpy as np, cv2
from ultralytics import YOLO
W = glob.glob("/kaggle/input/**/combined_640/weights/best.pt", recursive=True)[0]
NAMES = YOLO(W).names
print("Model:", W, NAMES)

vids = sorted(glob.glob("/kaggle/input/**/test/**/visible.mp4", recursive=True))
if not vids:
    vids = sorted(glob.glob("/kaggle/input/**/visible.mp4", recursive=True))
print(len(vids), "visible videos found, e.g.", vids[:2])
print(os.listdir(os.path.dirname(vids[0])))

In [ ]:
# 2. Load labels: Anti-UAV stores per-frame [x, y, w, h] in gt_rect and a 0/1 'exist' flag
def load_gt(video):
    d = os.path.dirname(video)
    cands = [os.path.join(d, n) for n in ["visible.json", "Visible.json", "visible_label.json"]]
    cands += glob.glob(os.path.join(d, "*isible*.json"))
    path = next(p for p in cands if os.path.exists(p))
    j = json.load(open(path))
    rects, exist = j["gt_rect"], j.get("exist", [1] * len(j["gt_rect"]))
    out = []
    for r, e in zip(rects, exist):
        if e and r and len(r) == 4 and r[2] > 0 and r[3] > 0:
            out.append(np.array([r[0], r[1], r[0] + r[2], r[1] + r[3]], float))
        else:
            out.append(None)
    return out

random.seed(0)
SEQS = vids[:: max(1, len(vids) // 20)][:20]     # 20 videos spread across the test split
MAX_FRAMES = 300                                  # first 300 frames (~12 s) of each
print("Using", len(SEQS), "videos")

In [ ]:
# 3. Run the setups
def iou1(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

MIN_TRACK = 5      # a low-confidence box needs a track this many frames old before it's reported
SURE_CONF = 0.5    # a box this confident is reported right away

# Tracker configs: start from Ultralytics' defaults and change a few settings
import yaml
from ultralytics.utils import ROOT
def tracker_cfg(base, name, **changes):
    cfg = yaml.safe_load(open(ROOT / "cfg" / "trackers" / base))
    cfg.update(changes)
    path = f"/kaggle/working/{name}.yaml"
    yaml.safe_dump(cfg, open(path, "w"))
    print(name, cfg)
    return path
BYTE_DEFAULT = tracker_cfg("bytetrack.yaml", "byte_default")
BYTE_TUNED = tracker_cfg("bytetrack.yaml", "byte_tuned", track_buffer=60, match_thresh=0.9, new_track_thresh=0.3)
BOTSORT_GMC = tracker_cfg("botsort.yaml", "botsort_gmc", track_buffer=60, match_thresh=0.9,
                          gmc_method="sparseOptFlow", with_reid=False)

def run(video, imgsz, tracker=None, hybrid=False):
    model = YOLO(W)                      # fresh model = fresh tracker state per video
    cap = cv2.VideoCapture(video)
    seen = collections.Counter()
    frames = []
    for f in range(MAX_FRAMES):
        ok, img = cap.read()
        if not ok: break
        if tracker:
            r = model.track(img, imgsz=imgsz, conf=0.1, persist=True, tracker=tracker, verbose=False)[0]
            boxes = []
            if r.boxes.id is not None:
                for b, c, i, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(),
                                      r.boxes.id.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                    seen[int(i)] += 1
                    if seen[int(i)] >= MIN_TRACK or (hybrid and s >= SURE_CONF):
                        boxes.append((b, NAMES[int(c)], int(i)))
        else:
            r = model.predict(img, imgsz=imgsz, conf=0.25, verbose=False)[0]
            boxes = [(b, NAMES[int(c)], None) for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy())]
        frames.append(boxes)
    cap.release()
    return frames

def score(frames, gt):
    tp = fp = fn = drone_lbl = 0; ids = set()
    for boxes, g in zip(frames, gt):
        hit = None
        if g is not None:
            ious = [iou1(b, g) for b, _, _ in boxes]
            if ious and max(ious) >= 0.3:
                hit = int(np.argmax(ious))
        if g is not None and hit is None: fn += 1
        if hit is not None:
            tp += 1; drone_lbl += boxes[hit][1] == "drone"
            if boxes[hit][2] is not None: ids.add(boxes[hit][2])
        fp += len(boxes) - (hit is not None)
    return dict(tp=tp, fp=fp, fn=fn, drone_lbl=drone_lbl, n_ids=len(ids))

SETUPS = [("Detector only", dict()),
          ("ByteTrack default, 5-frame rule (v1)", dict(tracker=BYTE_DEFAULT)),
          ("ByteTrack default + hybrid", dict(tracker=BYTE_DEFAULT, hybrid=True)),
          ("ByteTrack tuned + hybrid", dict(tracker=BYTE_TUNED, hybrid=True)),
          ("BoT-SORT camera-motion + hybrid", dict(tracker=BOTSORT_GMC, hybrid=True))]
results = {s[0]: [] for s in SETUPS}
for k, v in enumerate(SEQS):
    gt = load_gt(v)
    for name, kw in SETUPS:
        fr = run(v, 640, **kw)
        results[name].append(score(fr, gt[:len(fr)]))
    print(f"{k+1}/{len(SEQS)} done: {os.path.basename(os.path.dirname(v))}")

In [ ]:
# 4. Results table
import pandas as pd
rows = []
for name, per in results.items():
    tp = sum(r["tp"] for r in per); fp = sum(r["fp"] for r in per); fn = sum(r["fn"] for r in per)
    p, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    row = dict(setup=name, precision=round(p, 3), recall=round(rc, 3), F1=round(2*p*rc / max(p + rc, 1e-9), 3),
               false_boxes_per_100_frames=round(100 * fp / sum(r["tp"] + r["fn"] for r in per), 1),
               labelled_drone=round(sum(r["drone_lbl"] for r in per) / max(tp, 1), 3))
    if name != "Detector only":
        row["track_ids_per_video"] = round(np.mean([r["n_ids"] for r in per if r["tp"]]), 2)
    rows.append(row)
table = pd.DataFrame(rows)
print(table.to_string(index=False))
table.to_csv("/kaggle/working/tracking_results.csv", index=False)
json.dump({"videos": [os.path.basename(os.path.dirname(v)) for v in SEQS], "frames_per_video": MAX_FRAMES,
           "min_track": MIN_TRACK, "results": rows}, open("/kaggle/working/tracking_summary.json", "w"), indent=1)

In [ ]:
# 5. Demo video: the setup with the best F1 on one test video, with track IDs and the ground truth in white
best = table[table.setup != "Detector only"].sort_values("F1", ascending=False).iloc[0]   # best tracked setup
kw = dict(SETUPS)[best.setup]
per = results[best.setup]
demo = SEQS[int(np.argmax([r["tp"] - r["fp"] for r in per]))]
print("Demo:", best.setup, "on", demo)
gt = load_gt(demo)
fr = run(demo, 640, **kw)
cap = cv2.VideoCapture(demo)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("/kaggle/working/tracking_demo.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
for f, boxes in enumerate(fr):
    ok, img = cap.read()
    if not ok: break
    if gt[f] is not None:
        x1, y1, x2, y2 = map(int, gt[f]); cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), 1)
    for b, lbl, i in boxes:
        x1, y1, x2, y2 = map(int, b)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 220, 0), 2)
        cv2.putText(img, f"{lbl}" + (f" #{i}" if i is not None else ""), (x1, max(y1 - 8, 16)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 220, 0), 2)
    cv2.putText(img, f"{best.setup} | frame {f}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 255), 2)
    out.write(img)
out.release(); cap.release()
print("Saved /kaggle/working/tracking_demo.mp4")

In [ ]:
# 6. Chart: precision and recall per setup
import matplotlib.pyplot as plt
ax = table.set_index("setup")[["precision", "recall", "F1"]].plot.bar(figsize=(10, 4.5), rot=20)
ax.set_ylim(0, 1); ax.set_title("Anti-UAV videos: detector vs. detector + tracking"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig("/kaggle/working/tracking_chart.png", dpi=150); plt.show()
for p in glob.glob("/kaggle/working/*.pt"): os.remove(p)
print(os.listdir("/kaggle/working"))